In [0]:
df=spark.read.table("data_engineering_workshop.demo1.bronze_table")

In [0]:
df.dropDuplicates()
df.display()
df.count()

In [0]:
df = df.dropDuplicates(["InvoiceNo", "StockCode"])

In [0]:
df.count()

In [0]:
from pyspark.sql.functions import col, to_timestamp,trim,year,month,upper

df = df.dropna(subset=["CustomerID", "Description"])

#  Convert data types
df = df.withColumn("Quantity", col("Quantity").cast("int")) \
       .withColumn("UnitPrice", col("UnitPrice").cast("double"))

#  Convert InvoiceDate to timestamp
df = df.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))

#  Remove invalid data (negative values)
df = df.filter((col("Quantity") > 0) & (col("UnitPrice") > 0))

#  Create new column (Revenue)
df = df.withColumn("Revenue", col("Quantity") * col("UnitPrice"))

#  Rename column
df = df.withColumnRenamed("CustomerID", "Customer_Id")

# Trim text columns (remove spaces)
df = df.withColumn("Description", trim(col("Description")))

# Standardize Country (uppercase)
df = df.withColumn("Country", upper(col("Country")))

# Extract Year from date
df = df.withColumn("Year", year(col("InvoiceDate")))

# Extract Month from date
df = df.withColumn("Month", month(col("InvoiceDate")))


In [0]:


df.write.mode("overwrite").saveAsTable("data_engineering_workshop.demo1.silver_table")

spark.read.table("data_engineering_workshop.demo1.silver_table").show(5)